# EcoMove — Hourly rental demand & scenario planning

A reproducible business analytics case study developed from the original exploratory analysis, with stronger evaluation and a Streamlit frontend.

## Reproduce the analysis

Install `requirements-dev.txt`, then run this notebook from the repository or its `notebooks` directory. The frontend and notebook use the same pipeline.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.pipeline import TARGET, load_data, run_experiment

raw = load_data()
raw.head()

,Date,Year,Month,Hour,Weather_Outcast,Temperature_Celsius,Feeling_Temperature_Celsius,Humidity_Percent,Wind_Speed_kph,Season,Public_Holiday,Week_Day,Business_Day,Registered_Member_Count,Casual_User_Count,Rental_Bikes_Total
0,1/1/2012,2012,Jan,0,Clear,8.9,9,66,0,Winter,Yes,Sunday,No,43,5.0,48
1,1/1/2012,2012,Jan,1,Clear,8.9,7,66,9,Winter,Yes,Sunday,No,78,15.0,93
2,1/1/2012,2012,Jan,2,Clear,7.0,7,76,0,Winter,Yes,Sunday,No,59,16.0,75
3,1/1/2012,2012,Jan,3,Clear,6.1,6,81,0,Winter,Yes,Sunday,No,41,11.0,52
4,1/1/2012,2012,Jan,4,Clear,5.2,4,81,6,Winter,Yes,Sunday,No,8,0.0,8


## Data quality

Raw missingness is visible below. Fixed validity rules and learned training-only preprocessing are documented in `docs/methodology.md`.

In [2]:
raw.isna().sum().sort_values(ascending=False).to_frame("Missing values")

,Missing values
Casual_User_Count,27
Date,0
Year,0
Month,0
Hour,0
Weather_Outcast,0
Temperature_Celsius,0
Feeling_Temperature_Celsius,0
Humidity_Percent,0
Wind_Speed_kph,0


In [3]:
result = run_experiment(raw)
pd.DataFrame(result["audit"].items(), columns=["Check", "Count"])

,Check,Count
0,input_rows,8734
1,invalid_date_hour_or_target,0
2,duplicate_hours,0
3,Temperature_Celsius_missing_or_invalid,6
4,Humidity_Percent_missing_or_invalid,6
5,Wind_Speed_kph_missing_or_invalid,4
6,usable_rows,8734


## Exploratory patterns

These descriptive summaries use the full historical sample. Model selection uses only the separate validation partition.

In [4]:
result["data"].groupby("Hour")[TARGET].agg(["mean", "median", "count"]).round(2)

,mean,median,count
Hour,,,
0,64.63,49.0,365
1,40.13,24.0,364
2,26.70,13.0,363
3,13.33,7.0,355
4,7.25,7.0,360
5,25.01,24.5,364
6,94.32,100.5,364
7,267.15,288.5,364
8,454.76,558.0,364


In [5]:
result["data"].groupby("Weather_Outcast")[TARGET].agg(["mean", "count"]).round(2)

,mean,count
Weather_Outcast,,
Clear,253.66,5768
Heavy Rain or Snow,93.50,2
Light Rain or Snow,142.38,638
Mist and Cloudy,213.00,2326


In [6]:
pd.DataFrame([{ "Split": k, "Rows": len(result[k]), "Start": result[k].Date.min(), "End": result[k].Date.max()} for k in ["train", "validation", "test"]])

,Split,Rows,Start,End
0,train,5246,2012-01-01,2012-08-06
1,validation,1752,2012-08-07,2012-10-18
2,test,1736,2012-10-19,2012-12-31


## Model selection

Candidates are fitted only to training records. The validation selection rule is fixed in the pipeline.

In [7]:
result["validation_metrics"].round(3)

,Model,MAE,RMSE,R2
0,Random forest,51.013,81.791,0.884
1,Ridge regression,150.846,220.938,0.154
2,Mean baseline,194.152,249.674,-0.080


In [8]:
print("Selected model:", result["selected"])
result["test_metrics"].round(3)

Selected model: Random forest


,Model,MAE,RMSE,R2
0,Mean baseline,154.475,189.329,-0.014
1,Ridge regression,118.690,181.318,0.070
2,Random forest,63.674,91.138,0.765


## Held-out predictions

Inspect errors and minority-class performance, not only an overall fit statistic.

In [9]:
result["predictions"].head(15)

,Timestamp,Actual,Predicted,Residual
6998,2012-10-19 00:00:00,56,62.188453,-6.188453
6999,2012-10-19 01:00:00,16,37.958739,-21.958739
7000,2012-10-19 02:00:00,10,18.847518,-8.847518
7001,2012-10-19 03:00:00,5,10.207823,-5.207823
7002,2012-10-19 04:00:00,6,9.869331,-3.869331
7003,2012-10-19 05:00:00,36,31.526850,4.473150
7004,2012-10-19 06:00:00,131,107.027940,23.972060
7005,2012-10-19 07:00:00,154,283.908793,-129.908793
7006,2012-10-19 08:00:00,467,539.036365,-72.036365
7007,2012-10-19 09:00:00,389,293.580073,95.419927


## Interpretation

Read the measured results in `reports/summary.json` and the limitations in `docs/methodology.md`. These are offline case-study results, not evidence of achieved operational benefits. Run `python evaluate.py` to regenerate the published tables.